# 大模型资源与配置估算：从参数规模到集群预算

> **本章定位**：根据模型结构、训练状态与推理负载，以统一容量模型估算显存、GPU、CPU、内存、存储与网络需求。

> **章节边界**：本章属于训练与推理系统：资源基础，服务于立项与扩容前的资源规划；估算结果不替代目标硬件基准、稳定性测试或正式采购评审。

**本章总览**：以模型配置、执行负载和完成期限为输入，构建覆盖 GPU、CPU、内存、存储与网络的可校准容量模型，并输出假设、余量和复核条件。

```mermaid
flowchart LR
    A["模型配置"] --> P["参数量"]
    B["精度与训练方法"] --> M["显存模型"]
    C["Token 预算与期限"] --> F["计算量模型"]
    D["数据吞吐与制品策略"] --> H["CPU / RAM / 存储 / 网络"]
    P --> M
    P --> F
    M --> G["显存约束 GPU 数"]
    F --> T["期限约束 GPU 数"]
    G --> R["资源建议"]
    T --> R
    H --> R
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 训练与推理系统：资源基础；模型训练与适配的预算专题 |
| 本章定位 | 纵向建立可校准容量模型，把模型配置转换为完整资源预算。 |
| 先修知识 | 掌握 `31` 的模型结构、参数、上下文与生成过程；训练预算需理解梯度、优化器状态和 Token 工作量，相关知识见 `40`；服务预算需理解并发与请求长度。 |
| 预计时间 | 60～90 分钟 |
| 运行资源 | CPU 即可；最终结果必须用目标硬件基准校准。 |
| 输入 | 模型参数、精度、上下文、Batch、Token 预算、期限和硬件配置。 |
| 交付物 | GPU、CPU、RAM、存储、网络和期限约束的容量报告。 |

### 1.1．学习目标

完成本章后，读者能够从模型 Config、精度、上下文、并发、Token 预算和期限推导资源需求，区分显存约束与计算期限约束，并使用目标硬件实测结果校准容量假设。


## 2．直觉与输入输出契约

对使用 GQA/MQA 和门控 FFN 的 Decoder-only 模型，每层主要参数为：

$$
P_{attn}=d^2+2d(n_{kv}d_h)+d^2
$$

$$
P_{ffn}=3dd_{ff}
$$

其中第一项分别对应 Q、K/V、输出投影；门控 FFN 包含 gate、up、down 三个矩阵。词嵌入与 LM Head 是否共享会造成一个 $Vd$ 的差异。

![架构图：Decoder-only 模型各参数域到容量预算的映射](assets/figures/A10_resource_planning/model-parameter-map.svg)

[TikZ 源文件](assets/figures/A10_resource_planning/model-parameter-map.tex)


<!-- theory-math-contract:v1 -->
### 2.1．核心机制的语言与数学表达

资源规划先把模型权重、运行时状态和峰值余量分开估算。推理显存下界可写为：

$$
M_{\mathrm{total}}\approx M_{\mathrm{weights}}+M_{\mathrm{KV}}+M_{\mathrm{activations}}+M_{\mathrm{runtime}}+M_{\mathrm{margin}},
\qquad M_{\mathrm{weights}}=P\frac{b_w}{8}
$$

其中，$P$ 是参数量，$b_w$ 是每个权重的存储位数，各 $M$ 的单位必须统一为 Byte 或 GiB。容量模型函数对应各项求和，实测峰值用于回填校准系数。公式是预算起点而非采购结论；量化元数据、临时 Workspace、碎片、并行复制和不同 Kernel 都会使实测值偏离理论下界。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

本节将参数量、推理显存、训练显存、计算期限与主机资源分别实现为可复算函数。所有函数显式接收模型、负载、精度、硬件和余量参数，不依赖隐藏的设备默认值。


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class MyModelConfig:
    """描述 Dense Decoder 参数量估算所需的核心模型配置。"""
    vocab_size: int
    hidden_size: int
    intermediate_size: int
    num_hidden_layers: int
    num_attention_heads: int
    num_key_value_heads: int
    tie_word_embeddings: bool = True


def my_estimate_parameter_breakdown(config: MyModelConfig) -> dict[str, int]:
    """估算 Embedding、Attention、FFN、Norm 与 LM Head 的参数分项。"""
    head_dim = config.hidden_size // config.num_attention_heads
    embedding = config.vocab_size * config.hidden_size

    q_projection = config.hidden_size * config.hidden_size
    kv_projections = 2 * config.hidden_size * config.num_key_value_heads * head_dim
    output_projection = config.hidden_size * config.hidden_size
    attention_per_layer = q_projection + kv_projections + output_projection

    # SwiGLU 一类门控 FFN 包含 gate、up、down 三个投影。
    ffn_per_layer = 3 * config.hidden_size * config.intermediate_size
    norm_per_layer = 2 * config.hidden_size
    transformer = config.num_hidden_layers * (attention_per_layer + ffn_per_layer + norm_per_layer)
    final_norm = config.hidden_size
    lm_head = 0 if config.tie_word_embeddings else embedding

    return {
        "embedding": embedding,
        "transformer": transformer,
        "final_norm": final_norm,
        "lm_head": lm_head,
        "total": embedding + transformer + final_norm + lm_head,
    }


# 该组字段构造约 7B 级 Dense Decoder；生产必须读取目标 Config 并复核 hidden/head 整除与 Embedding 共享。
model_7b_like = MyModelConfig(
    vocab_size=32_000,       # 词表增大将线性增加 Embedding，数值须绑定 Tokenizer。
    hidden_size=4_096,       # 模型宽度会同时放大 Attention、FFN、激活和 KV Cache。
    intermediate_size=11_008,  # FFN 宽度与 hidden 耦合，改变后按目标架构重算参数量。
    num_hidden_layers=32,    # 层数近似线性放大参数、激活与 KV Cache。
    num_attention_heads=32,  # 4096÷32 定义 head_dim；必须保持整除。
    num_key_value_heads=32,  # 标准 MHA；改小为 GQA 会降低 K/V 参数和缓存，但须与 Checkpoint 一致。
)

parameter_breakdown = my_estimate_parameter_breakdown(model_7b_like)
{key: f"{value / 1e9:.3f} B" for key, value in parameter_breakdown.items()}


### 3.1．推理显存模型

推理显存至少包含：

$$
M_{inference}=M_{weights}+M_{KV}+M_{workspace}+M_{reserve}
$$

单个序列的 KV Cache 近似为：

$$
M_{KV}=2LBTn_{kv}d_hb_{kv}
$$

其中 `2` 表示 K 和 V，$B$ 是并发序列数，$T$ 是缓存长度。GQA/MQA 主要通过降低 $n_{kv}$ 减少 KV Cache。Paged KV、Prefix Cache、量化 KV 和跨卡切分会改变实际峰值，但不改变容量分析的分项方法。


In [ ]:
import math

GIB = 1024 ** 3


def my_estimate_inference_memory(
    config: MyModelConfig,
    parameter_count: int,
    weight_bits: int,
    batch_sequences: int,
    cached_tokens_per_sequence: int,
    kv_bytes: int = 2,  # BF16/FP16 KV 每元素 2 Byte；KV dtype 变化后按实际存储位宽重算。
    quantization_metadata_ratio: float = 0.06,  # 暂估量化元数据占 6%；分组与 Kernel 变化后用制品大小校准。
    workspace_ratio: float = 0.10,  # 暂留 10% 工作区；后端、CUDA Graph 或并发变化后按峰值显存反推。
) -> dict[str, float]:
    """估算权重、量化元数据、KV Cache 和工作区的推理显存。"""
    head_dim = config.hidden_size // config.num_attention_heads
    raw_weight_bytes = parameter_count * weight_bits / 8
    metadata_bytes = raw_weight_bytes * quantization_metadata_ratio if weight_bits < 16 else 0.0
    kv_cache_bytes = (
        2
        * config.num_hidden_layers
        * batch_sequences
        * cached_tokens_per_sequence
        * config.num_key_value_heads
        * head_dim
        * kv_bytes
    )
    workspace_bytes = (raw_weight_bytes + metadata_bytes + kv_cache_bytes) * workspace_ratio
    total_bytes = raw_weight_bytes + metadata_bytes + kv_cache_bytes + workspace_bytes
    return {
        "weights_gib": raw_weight_bytes / GIB,
        "quant_metadata_gib": metadata_bytes / GIB,
        "kv_cache_gib": kv_cache_bytes / GIB,
        "workspace_gib": workspace_bytes / GIB,
        "total_before_reserve_gib": total_bytes / GIB,
    }


inference_memory_contract = {
    "config": model_7b_like,
    "parameter_count": parameter_breakdown["total"],
    "weight_bits": 16,
    "batch_sequences": 8,  # 同时维护 8 条序列；并发翻倍会近似线性增加 KV Cache。
    "cached_tokens_per_sequence": 4_096,  # 每序列缓存 4096 Token；长度分布或调度策略变化即重算。
}
inference_estimate = my_estimate_inference_memory(**inference_memory_contract)
inference_estimate


### 3.2．训练显存模型

经典 BF16 + AdamW 全参数训练常见模型状态近似为每参数 16 字节：BF16 权重 2、BF16 梯度 2、FP32 Master Weight 4、两个 FP32 Moment 8。实现细节、低精度优化器和融合内核会改变这个数字。

ZeRO/FSDP 的关键是分别决定哪些状态被数据并行 Rank 分片：

| 策略 | 优化器状态 | 梯度 | 参数 |
|---|---:|---:|---:|
| DDP | 不分片 | 不分片 | 不分片 |
| ZeRO-1 | 分片 | 不分片 | 不分片 |
| ZeRO-2 | 分片 | 分片 | 不分片 |
| ZeRO-3 / FSDP Full Shard | 分片 | 分片 | 分片 |

激活与模型结构、序列长度、Micro-batch、Attention 实现和重计算策略有关。本章将它作为“每层每 Token 保存多少个 hidden-size 元素”的可校准系数，而不是伪装成精确常量。


In [ ]:
from typing import Literal


@dataclass(frozen=True)
class MyTrainingMemoryConfig:
    """描述训练并行、微批、序列、激活与 ZeRO 策略假设。"""
    data_parallel_size: int
    micro_batch_size: int
    sequence_length: int
    activation_elements_per_token_layer: float
    activation_checkpointing_ratio: float
    strategy: Literal["ddp", "zero1", "zero2", "zero3"]


def my_estimate_training_memory(
    model: MyModelConfig,
    parameter_count: int,
    training: MyTrainingMemoryConfig,
) -> dict[str, float]:
    """按分片策略估算单卡权重、梯度、优化器和激活显存。"""
    dp = training.data_parallel_size
    optimizer_shard = dp if training.strategy in {"zero1", "zero2", "zero3"} else 1
    gradient_shard = dp if training.strategy in {"zero2", "zero3"} else 1
    parameter_shard = dp if training.strategy == "zero3" else 1

    weight_bytes = parameter_count * 2 / parameter_shard
    gradient_bytes = parameter_count * 2 / gradient_shard
    # FP32 Master Weight 与两个 FP32 Moment 共 12 字节。
    optimizer_bytes = parameter_count * 12 / optimizer_shard
    activation_bytes = (
        training.micro_batch_size
        * training.sequence_length
        * model.num_hidden_layers
        * model.hidden_size
        * training.activation_elements_per_token_layer
        * 2
        * training.activation_checkpointing_ratio
    )
    total_bytes = weight_bytes + gradient_bytes + optimizer_bytes + activation_bytes
    return {
        "weights_gib_per_gpu": weight_bytes / GIB,
        "gradients_gib_per_gpu": gradient_bytes / GIB,
        "optimizer_gib_per_gpu": optimizer_bytes / GIB,
        "activations_gib_per_gpu": activation_bytes / GIB,
        "total_before_workspace_gib_per_gpu": total_bytes / GIB,
    }


training_memory_contract = MyTrainingMemoryConfig(
    data_parallel_size=8,  # 8 路 DP 同时决定 ZeRO 分片因子，须与集群拓扑一致。
    micro_batch_size=1,  # 单卡微批为 1 以容纳长序列；增大会近似线性增加激活。
    sequence_length=4_096,  # 4096 Token 用于长上下文规划；长度变化后重测 Attention 与激活峰值。
    activation_elements_per_token_layer=10.0,  # 首轮假设；用目标模型峰值测量反推。
    activation_checkpointing_ratio=0.35,       # 规划假设：重计算后仅保留约 35% 的激活。
    strategy="zero3",  # ZeRO-3 分片参数、梯度和优化器状态；通信、恢复格式或拓扑变化时重新选型。
)
training_memory = my_estimate_training_memory(
    model=model_7b_like,
    parameter_count=parameter_breakdown["total"],
    training=training_memory_contract,
)
training_memory


#### 3.2.1．推理上下文与训练分片的显存构成

**学习问题**：同一模型在推理时延长上下文、在训练时切换 DDP/ZeRO 分片策略，分别改变了哪些单卡显存分项？

**验收不变量**：固定模型、精度和 Batch 后，权重显存必须与上下文长度无关，KV Cache 在缓存长度翻倍时必须翻倍；固定 Micro-batch、序列长度和激活重计算比例后，切换分片策略不能改变激活估算，ZeRO-1/2/3 应依次按表中契约分片优化器、梯度和参数。


In [ ]:
# 直接复用本章推理与训练显存函数，展示上下文敏感性和分片策略的分项影响。
import matplotlib.pyplot as plt

inference_context_lengths = [512, 1_024, 2_048, 4_096]
inference_memory_rows = [
    my_estimate_inference_memory(
        **{
            **inference_memory_contract,
            "cached_tokens_per_sequence": context_length,
        }
    )
    for context_length in inference_context_lengths
]

training_strategies = ["ddp", "zero1", "zero2", "zero3"]
training_memory_rows = {}
for strategy in training_strategies:
    strategy_contract = MyTrainingMemoryConfig(
        data_parallel_size=training_memory_contract.data_parallel_size,
        micro_batch_size=training_memory_contract.micro_batch_size,
        sequence_length=training_memory_contract.sequence_length,
        activation_elements_per_token_layer=training_memory_contract.activation_elements_per_token_layer,
        activation_checkpointing_ratio=training_memory_contract.activation_checkpointing_ratio,
        strategy=strategy,
    )
    training_memory_rows[strategy] = my_estimate_training_memory(
        model=model_7b_like,
        parameter_count=parameter_breakdown["total"],
        training=strategy_contract,
    )


def my_plot_stacked_memory(axis, labels, rows, components):
    """在指定坐标轴绘制显存分项堆叠柱并标注总量。"""
    bottoms = [0.0] * len(rows)
    for key, display_name, color in components:
        values = [row[key] for row in rows]
        axis.bar(labels, values, bottom=bottoms, label=display_name, color=color)
        bottoms = [bottom + value for bottom, value in zip(bottoms, values)]
    for index, total in enumerate(bottoms):
        axis.text(index, total, f"{total:.1f}", ha="center", va="bottom", fontsize=8)
    axis.set_ylabel("单卡估算显存（GiB）")
    axis.grid(axis="y", alpha=0.25)
    axis.margins(y=0.15)


fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
my_plot_stacked_memory(
    axes[0],
    [f"{length:,}" for length in inference_context_lengths],
    inference_memory_rows,
    [
        ("weights_gib", "权重", "#0072B2"),
        ("kv_cache_gib", "KV Cache", "#E69F00"),
        ("workspace_gib", "工作区", "#009E73"),
    ],
)
axes[0].set_xlabel("每序列已缓存 Token")
axes[0].set_title("BF16 推理：Batch=8，只有 KV 与相关工作区随上下文增长")
axes[0].legend()

training_rows_in_order = [training_memory_rows[strategy] for strategy in training_strategies]
my_plot_stacked_memory(
    axes[1],
    ["DDP", "ZeRO-1", "ZeRO-2", "ZeRO-3"],
    training_rows_in_order,
    [
        ("weights_gib_per_gpu", "权重", "#0072B2"),
        ("gradients_gib_per_gpu", "梯度", "#56B4E9"),
        ("optimizer_gib_per_gpu", "优化器状态", "#E69F00"),
        ("activations_gib_per_gpu", "激活", "#009E73"),
    ],
)
axes[1].set_xlabel("8 路数据并行策略")
axes[1].set_title("BF16 + AdamW 训练：分片对象决定单卡模型状态")
axes[1].legend()
fig.suptitle("显存不是单一数字：负载长度与分片策略改变不同组成项", fontsize=14)
fig.tight_layout()
plt.show()

ddp_memory = training_memory_rows["ddp"]
zero1_memory = training_memory_rows["zero1"]
zero2_memory = training_memory_rows["zero2"]
zero3_memory = training_memory_rows["zero3"]
memory_composition_checks = {
    "inference_weights_constant": len({row["weights_gib"] for row in inference_memory_rows}) == 1,
    "kv_doubles_with_context": all(
        math.isclose(right["kv_cache_gib"], 2 * left["kv_cache_gib"])
        for left, right in zip(inference_memory_rows, inference_memory_rows[1:])
    ),
    "activations_independent_of_sharding": len(
        {row["activations_gib_per_gpu"] for row in training_rows_in_order}
    ) == 1,
    "zero1_optimizer_sharded_8_way": math.isclose(
        zero1_memory["optimizer_gib_per_gpu"], ddp_memory["optimizer_gib_per_gpu"] / 8
    ),
    "zero2_gradient_sharded_8_way": math.isclose(
        zero2_memory["gradients_gib_per_gpu"], ddp_memory["gradients_gib_per_gpu"] / 8
    ),
    "zero3_weights_sharded_8_way": math.isclose(
        zero3_memory["weights_gib_per_gpu"], ddp_memory["weights_gib_per_gpu"] / 8
    ),
}
print(memory_composition_checks)


**应观察结论**：左图中权重部分保持不变，KV Cache 随已缓存 Token 数翻倍而翻倍；右图中 ZeRO-1 首先降低优化器状态，ZeRO-2 继续分片梯度，ZeRO-3 再分片参数，而激活在本次仅改变分片策略的比较中保持不变。

**不可误读边界**：堆叠柱表示容量函数的单卡估算，不是目标设备的峰值测量。Attention Kernel、临时张量、通信缓冲、Allocator 碎片、参数 All-gather 峰值和安全余量仍需实测；较低的 ZeRO 单卡状态也不自动等于更高吞吐或更少集群总显存。


### 3.3．显存与期限的联合约束

训练计算量的首轮近似：

$$
F_{train} ≈ 6PT
$$

给定单卡峰值吞吐 $F_{peak}$、MFU $u$、期限 $S$ 秒，计算约束卡数为：

$$
N_{compute} = ceil((6PT) / (F_{peak}uS))
$$

最终卡数至少为显存约束与计算约束的最大值，还要调整为 TP/PP/EP 拓扑可整除的数量，并保留故障与维护容量。


In [ ]:
@dataclass(frozen=True)
class MyAcceleratorProfile:
    """保存加速器显存、峰值算力及可用率规划假设。"""
    memory_gib: float
    bf16_peak_tflops: float
    usable_memory_ratio: float = 0.85  # 仅使用标称显存的 85%，其余留给碎片和运行时；按 P99 峰值校准。
    expected_mfu: float = 0.35  # 35% MFU 是未校准起点；须用目标精度、Kernel 与拓扑的稳定窗口实测。


def my_estimate_gpu_count(
    total_memory_gib: float,
    parameter_count: int,
    training_tokens: int,
    deadline_days: float,
    accelerator: MyAcceleratorProfile,
    topology_multiple: int = 8,  # 按 8 卡节点示例取整；实际须满足节点 GPU 数及 TP/PP/EP 整除关系。
) -> dict[str, int]:
    """综合显存、计算期限和拓扑整除约束估算 GPU 数量。"""
    usable_memory_gib = accelerator.memory_gib * accelerator.usable_memory_ratio
    memory_bound = math.ceil(total_memory_gib / usable_memory_gib)

    training_flops = 6 * parameter_count * training_tokens
    deadline_seconds = deadline_days * 24 * 60 * 60
    useful_flops_per_gpu_second = accelerator.bf16_peak_tflops * 1e12 * accelerator.expected_mfu
    compute_bound = math.ceil(training_flops / (useful_flops_per_gpu_second * deadline_seconds))

    raw_count = max(memory_bound, compute_bound)
    topology_adjusted = math.ceil(raw_count / topology_multiple) * topology_multiple
    return {
        "memory_bound_gpus": memory_bound,
        "compute_bound_gpus": compute_bound,
        "topology_adjusted_gpus": topology_adjusted,
    }


# 80 GiB 与 312 TFLOP/s 构成可复算硬件画像，不代表任意 GPU 或实效吞吐。
accelerator = MyAcceleratorProfile(memory_gib=80, bf16_peak_tflops=312)
gpu_plan = my_estimate_gpu_count(
    total_memory_gib=training_memory["total_before_workspace_gib_per_gpu"] * 8,
    parameter_count=parameter_breakdown["total"],
    training_tokens=300_000_000_000,  # 300B Token 是项目预算；增加预算会线性提高总计算量。
    deadline_days=30,  # 30 天期限缩短时，计算约束卡数反向增加。
    accelerator=accelerator,
)
gpu_plan


### 3.4．CPU、内存、存储与网络

GPU 数量只是方案的一部分。生产规划还要回答：

- **CPU 核数**：以目标解析/分词吞吐除以单核实测吞吐，再乘余量；不按 GPU 数量作无依据的线性推断。
- **主机内存**：至少容纳 Checkpoint 暂存、DataLoader 队列、对象存储上传缓冲和恢复窗口。
- **存储**：原始、清洗后、Tokenized Shard、Checkpoint、评测和日志要分别预算，还需覆盖中间资产与恢复副本。
- **网络**：训练通信与数据读取分开评估；有效吞吐要计入协议、拥塞和并发折损。

![架构图：存储、CPU、加速器、集群通信与运维系统的数据通路](assets/figures/A10_resource_planning/resource-data-path.svg)

[TikZ 源文件](assets/figures/A10_resource_planning/resource-data-path.tex)


In [ ]:
def my_estimate_host_and_storage(
    target_preprocess_tokens_per_second: float,
    measured_tokens_per_second_per_core: float,
    checkpoint_gib: float,
    checkpoint_host_copies: int,
    dataloader_buffer_gib: float,
    raw_corpus_tib: float,
    cleaned_ratio: float,
    tokenized_ratio_of_raw: float,
    retained_checkpoints: int,
    headroom_ratio: float = 1.2,  # 主机和存储预留 20%；按 P95/P99 峰值、故障演练与滚动发布复核。
) -> dict[str, float]:
    """估算数据预处理 CPU、主机内存与多阶段制品存储容量。"""
    cpu_cores = math.ceil(
        target_preprocess_tokens_per_second / measured_tokens_per_second_per_core * headroom_ratio
    )
    host_ram_gib = (
        checkpoint_gib * checkpoint_host_copies + dataloader_buffer_gib
    ) * headroom_ratio
    storage_tib = (
        raw_corpus_tib
        + raw_corpus_tib * cleaned_ratio
        + raw_corpus_tib * tokenized_ratio_of_raw
        + checkpoint_gib * retained_checkpoints / 1024
    ) * headroom_ratio
    return {
        "cpu_cores": cpu_cores,
        "host_ram_gib": host_ram_gib,
        "storage_tib": storage_tib,
    }


host_plan = my_estimate_host_and_storage(
    target_preprocess_tokens_per_second=2_000_000,  # 目标 2M Token/s；交付期限或数据规模变化后重算。
    measured_tokens_per_second_per_core=40_000,  # 40k Token/s/core 仅为夹具，须替换为目标 Tokenizer 单核基准。
    checkpoint_gib=130,  # 单份 130 GiB；模型、优化器或保存精度变化后读取真实制品大小。
    checkpoint_host_copies=2,  # 当前写入与上一可恢复副本；恢复策略变化后调整。
    dataloader_buffer_gib=64,  # 64 GiB 缓冲是假设，按预取深度与峰值 RSS 校准。
    raw_corpus_tib=10,  # 10 TiB 原始语料是规划输入，不是通用数据规模。
    cleaned_ratio=0.55,  # 清洗制品占原始量 55% 的假设，按实际压缩与过滤率回填。
    tokenized_ratio_of_raw=0.45,  # Token 化制品占原始量 45% 的假设，Tokenizer/格式变化即重测。
    retained_checkpoints=5,  # 保留 5 份用于容量演示；按恢复窗口与对象存储成本复核。
)
host_plan


### 3.5．7B 与 70B 的量级参考

下表只按参数位宽计算权重或训练状态下界，单位为 GiB；它不包含量化元数据、KV Cache、激活、临时工作区、通信缓冲和安全余量。

| 参数规模 | BF16 权重 | INT8 权重 | INT4 权重 | BF16 AdamW 全参数训练状态（约 16 Byte/参数） |
|---:|---:|---:|---:|---:|
| 7B | 13.04 | 6.52 | 3.26 | 104.31 |
| 70B | 130.39 | 65.19 | 32.60 | 1043.08 |

因此，“7B 能否放进一张 16 GiB 卡”在仅权重 BF16 推理和长上下文高并发推理中会得到不同答案；全参数训练还必须结合 ZeRO/FSDP 分片、激活重计算、序列长度、Micro-batch 和通信拓扑。上表适合做数量级初筛，最终卡数由前面的显存约束和计算期限两者取大后，再按服务器与网络拓扑取整。


### 3.6．任务类型与显存口径

| 任务 | 主要显存项 | 第一约束 | 实测指标 |
|---|---|---|---|
| BF16 推理 | 权重 + KV + 工作区 | 并发和上下文长度 | TTFT、TPOT、峰值 KV、批调度效率 |
| 4-bit 推理 | 量化权重 + 元数据 + KV | Kernel 支持和 KV | 实际压缩率、反量化工作区、吞吐 |
| LoRA/QLoRA 微调 | 冻结权重 + 激活 + Adapter/优化器 | 激活和量化后端 | 峰值显存、可训练参数、Checkpoint 大小 |
| 全参数预训练 | 权重 + 梯度 + 优化器 + 激活 | 显存、计算期限、通信 | MFU、重计算比例、Collective 时间 |

固定模型规模不能唯一确定设备数量；容量结论需要同时给出精度、上下文长度、Batch、训练方法、硬件可用显存、MFU、期限和余量。


## 4．证据验证

容量模型需要通过目标硬件上的测量校准：参数量与权重清单相互核对；权重加载、峰值显存、每 Token 激活和 KV Cache 在最小拓扑上实测；目标序列长度与 Micro-batch 用于建立 OOM 边界；稳定窗口用于计算 MFU、数据等待和通信占比。实测值回填校准系数后，再以基准、保守和压力场景验证估算误差。

验收结果同时保留公式输入、测量窗口、硬件拓扑、预测值、实测值和相对误差。资源公式不因预算约束而改变；超出预算时应调整工作负载、训练方案、硬件或期限，并重新评估质量与风险。


## 5．迁移到生产库

生产路径从模型仓库当前提供的 Transformers Config 读取词表、隐藏维度、层数、注意力头、KV Head、FFN 维度、权重共享与数据类型，再与目标硬件基准、推理后端和训练并行配置组合。原理函数仍作为容量下界与回归基线，不承担硬件测量职责。

| 原理对象 | 生产输入 | 对齐证据 |
|---|---|---|
| 参数量公式 | Transformers Config 与权重清单 | 分层参数统计及共享权重差异 |
| 权重与 KV 容量 | 推理后端配置、并发和长度分布 | 峰值显存、KV dtype 与工作区 |
| 训练状态与激活 | 训练引擎、精度和并行策略 | 分片状态、重计算比例与峰值显存 |
| 期限模型 | 硬件吞吐基准与稳定 MFU | 总 Token、有效 FLOP/s 与完成时间 |


## 6．生产边界

资源报告应明确模型 ID、关键文件哈希、数据版本、精度、上下文和并发分布、训练方法、硬件 SKU、拓扑、后端、测量窗口、余量是否重复计算以及校准误差。GPU 数量同时受显存与完成期限约束，CPU、内存、存储和网络也需要独立容量与故障余量。

发布前应覆盖故障 Rank、Checkpoint 恢复、存储降速和网络拥塞演练。最终交付物不是单一 GPU 数量，而是一份可审计、可校准、包含基准与压力场景并注明复测条件的容量模型。
